# Notebook Dedicated to create the list of candidates (GTID + runID + subrunID)

In [2]:
import numpy as np
import glob
import pickle
import pandas as pd
import seaborn as sn

# Load Data

In [3]:
# ------- Observable list -------
obs_list = ['energy_corrected', 'posr_av', 'itr', 'eventID', 'runID']

# ------- Observable Dictionary -------
obs_dict = {obs: np.array([]) for obs in obs_list}

# ------- Directory of data -------
candidates_data_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/resume_files/'
hs_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/HS_results/pkl_resume/'

# Loop over the observable list to load the candidates data in obs_dict
for obs in obs_list:
    
    obs_dir = candidates_data_dir + obs + '.npy'
    obs_i = np.load(obs_dir)
    obs_dict[obs] = np.append(obs_dict[obs], obs_i)

print(f'Nº of intial events : {len(obs_dict['energy_corrected'])}')

# ===== Apply General Cuts =====
en_inf_cut = 5
posr_cut = 5500

energy_condition = (obs_dict['energy_corrected'] >= en_inf_cut)
posr_condition = (obs_dict['posr_av'] <= posr_cut)
runID_condition = (obs_dict['runID'] > 301000)
itr_condition = (obs_dict['itr'] >= 0.20)

mask = (energy_condition & posr_condition & runID_condition & itr_condition)

energy = obs_dict['energy_corrected'][mask]
posr_av = obs_dict['posr_av'][mask]
runID = obs_dict['runID'][mask]
itr = obs_dict['itr'][mask]
eventID = obs_dict['eventID'][mask]

print(f'Nº of events after basic cuts: {len(energy)}')

# ===== Now Apply Hotspot Cuts =====

# Load HS results
with open(hs_dir + 'hs_prompt_resume.pkl', 'rb') as f:
    hs_prompt_dict = pickle.load(f)
    
    hs_prompt_eventID = hs_prompt_dict['eventID']
    hs_prompt_runID = hs_prompt_dict['runID']
    
with open(hs_dir + 'hs_delay_resume.pkl', 'rb') as f:
    hs_delay_dict = pickle.load(f)
    
    hs_delay_eventID = hs_delay_dict['eventID']
    hs_delay_runID = hs_delay_dict['runID']

hs_eventID = np.concatenate((hs_prompt_eventID, hs_delay_eventID))
hs_runID = np.concatenate((hs_prompt_runID, hs_delay_runID))

# Remove coincident eventID and runID events with tagged HS eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs = (hs_runID.astype(np.int64) * offset) + hs_eventID.astype(np.int64)

is_hotspot = np.isin(unique_id_data, unique_id_hs)

hs_cut = ~is_hotspot

# Select the data
energy = energy[hs_cut]
posr_av = posr_av[hs_cut]
runID = runID[hs_cut]
itr = itr[hs_cut]
eventID = eventID[hs_cut]

print(f'Nº of events after removing HS events: {len(energy)}')

Nº of intial events : 33543
Nº of events after basic cuts: 174
Nº of events after removing HS events: 173


# Save Files

In [7]:
fdir_save = 'candidates list/'

## Create the Excel Folder

### All files together (Analysis15, Analysis15_bMR, Analysis20_bMR)

In [14]:
f_outname = f'filtered_solar_analysis_E_cut_{en_cut}_MeV_R_cut_5500_mm'  # Output file name

df = pd.DataFrame({'eventID': np.array(eventID, dtype = np.int64),
                   'runID': runID,
                   'subrunID': subrunID})

df.to_excel(f_outname + '.xlsx', index = False)

## Create txt list

In [8]:
data = np.column_stack((runID, eventID))
np.savetxt(fdir_save + f"filtered_solar_analysis_E_cut_{en_inf_cut}_MeV_R_cut_{posr_cut}_mm.txt", data,
           fmt="%d, %d",          # enteros
           delimiter=", ",
           comments="")